## 1. Instalacja i konfiguracja

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install librosa==0.11.0
%pip install mirdata==0.3.9
%pip install matplotlib==3.9.4
%pip install numpy==1.26.3
%pip install scikit-learn==1.6.1
%pip install tqdm==4.67.1
%pip install pretty_midi>=0.2.10
%pip install madmom==0.18.0
%pip install pypianoroll==1.0.2

In [ ]:
# Podstawowe biblioteki
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
from enum import Enum
import math

# Przetwarzanie dźwięku
import mirdata  # Dataset GuitarSet
from tqdm import tqdm

# Uczenie maszynowe
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


# Konfiguracja GPU - automatyczne wykrywanie
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Używane urządzenie: {device}")
if device.type == 'cuda':
    print(f"Model karty GPU: {torch.cuda.get_device_name(0)}")

## 2. Ładowanie i przygotowanie danych 

### 2.1. Inicjalizacja guitarset

In [ ]:
def initialize_guitarset(data_dir="guitarset_data"):
    """Initialize and download GuitarSet if needed"""
    # Utwórz folder jeśli nie istnieje
    os.makedirs(data_dir, exist_ok=True)
    
    # Inicjalizacja datasetu
    guitarset = mirdata.initialize("guitarset", data_home=data_dir)
    
    try:
        _ = guitarset.track_ids
    except FileNotFoundError:
        print("Pobieranie indeksu GuitarSet...")
        guitarset.download(partial_download=["index"])
    
    # Teraz sprawdź czy dane audio istnieją
    example_track = guitarset.track(guitarset.track_ids[0])
    if not os.path.exists(example_track.audio_mic_path):
        print("Pobieranie danych audio GuitarSet...")
        guitarset.download(partial_download=["audio"])
    else:
        print("GuitarSet już istnieje")
    
    return guitarset

guitarset = initialize_guitarset()

### 2.2. Stałe wartości

In [ ]:
FFT_HOP = 256
N_FFT = 8 * FFT_HOP

NOTES_BINS_PER_SEMITONE = 1
CONTOURS_BINS_PER_SEMITONE = 3
# base frequency of the CENTRAL bin of the first semitone (i.e., the
# second bin if annotations_bins_per_semitone is 3)
ANNOTATIONS_BASE_FREQUENCY = 27.5  # lowest key on a piano
ANNOTATIONS_N_SEMITONES = 88  # number of piano keys
AUDIO_SAMPLE_RATE = 22050
AUDIO_N_CHANNELS = 1
N_FREQ_BINS_NOTES = ANNOTATIONS_N_SEMITONES * NOTES_BINS_PER_SEMITONE
N_FREQ_BINS_CONTOURS = ANNOTATIONS_N_SEMITONES * CONTOURS_BINS_PER_SEMITONE

AUDIO_WINDOW_LENGTH = 2  # duration in seconds of training examples - original 1

ANNOTATIONS_FPS = AUDIO_SAMPLE_RATE // FFT_HOP
ANNOTATION_HOP = 1.0 / ANNOTATIONS_FPS

# ANNOT_N_TIME_FRAMES is the number of frames in the time-frequency representations we compute
ANNOT_N_FRAMES = ANNOTATIONS_FPS * AUDIO_WINDOW_LENGTH

# AUDIO_N_SAMPLES is the number of samples in the (clipped) audio that we use as input to the models
AUDIO_N_SAMPLES = AUDIO_SAMPLE_RATE * AUDIO_WINDOW_LENGTH - FFT_HOP

DATASET_SAMPLING_FREQUENCY = {
    "MAESTRO": 5,
    "GuitarSet": 2,
    "MedleyDB-Pitch": 2,
    "iKala": 2,
    "slakh": 2,
}


def _freq_bins(bins_per_semitone: int, base_frequency: float, n_semitones: int) -> np.array:
    d = 2.0 ** (1.0 / (12 * bins_per_semitone))
    bin_freqs = base_frequency * d ** np.arange(bins_per_semitone * n_semitones)
    return bin_freqs


FREQ_BINS_NOTES = _freq_bins(NOTES_BINS_PER_SEMITONE, ANNOTATIONS_BASE_FREQUENCY, ANNOTATIONS_N_SEMITONES)
FREQ_BINS_CONTOURS = _freq_bins(CONTOURS_BINS_PER_SEMITONE, ANNOTATIONS_BASE_FREQUENCY, ANNOTATIONS_N_SEMITONES)


class Split(Enum):
    train = "train"
    validation = "validation"
    test = "test"

### 2.2. Wczytanie danych

In [ ]:
def load_guitarset_track(track, resample=True):
    """Wczytaj ścieżkę audio i adnotacje z GuitarSeta i przygotuj do dalszego przetwarzania."""
    
    # Wczytaj audio
    audio, sr = librosa.load(track.audio_mic_path, sr=None, mono=True)
    
    # Resampluj jeśli potrzeba
    if resample and sr != AUDIO_SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=AUDIO_SAMPLE_RATE)
        sr = AUDIO_SAMPLE_RATE

    duration = librosa.get_duration(y=audio, sr=sr)
    time_scale = np.arange(0, duration + ANNOTATION_HOP, ANNOTATION_HOP)
    n_time_frames = len(time_scale)
    
    # Sparse indeksy dla adnotacji
    note_indices, note_values = track.notes_all.to_sparse_index(
        time_scale, "s", FREQ_BINS_NOTES, "hz"
    )
    onset_indices, onset_values = track.notes_all.to_sparse_index(
        time_scale, "s", FREQ_BINS_NOTES, "hz", onsets_only=True
    )
    contour_indices, contour_values = track.multif0.to_sparse_index(
        time_scale, "s", FREQ_BINS_CONTOURS, "hz"
    )
    
    return {
        "track_id": track.track_id,
        "audio": audio,
        "sr": sr,
        "note_indices": note_indices,
        "note_values": note_values,
        "onset_indices": onset_indices,
        "onset_values": onset_values,
        "contour_indices": contour_indices,
        "contour_values": contour_values,
        "n_time_frames": n_time_frames,
    }
# Wczytaj pierwszy utwór
example_track = guitarset.track(guitarset.track_ids[0])
track_data = load_guitarset_track(example_track)

# Sprawdź co mamy
print(f"Audio shape: {track_data['audio'].shape}")
print(f"Note indices: {track_data['note_indices'].shape}")
print(f"Contour indices: {track_data['contour_indices'].shape}")

### 2.3. Wizualizacja wczytanego nagrania

In [ ]:
def plot_cqt(audio, sr=AUDIO_SAMPLE_RATE, hop_length=FFT_HOP):
    C = librosa.cqt(audio, sr=sr, hop_length=hop_length, n_bins=88, bins_per_octave=12)
    C_db = librosa.amplitude_to_db(np.abs(C), ref=np.max)
    
    times = librosa.frames_to_time(np.arange(C_db.shape[1]), sr=sr, hop_length=hop_length)
    freqs = librosa.cqt_frequencies(n_bins=88, fmin=ANNOTATIONS_BASE_FREQUENCY, bins_per_octave=12)

    plt.figure(figsize=(15, 4))
    plt.title("Widmo CQT (log-amplituda)")
    plt.imshow(C_db, aspect="auto", origin="lower",
               extent=[times[0], times[-1], freqs[0], freqs[-1]],
               cmap="magma")
    plt.ylabel("Częstotliwość [Hz]")
    plt.xlabel("Czas [s]")
    plt.colorbar(label="Amplituda (dB)")
    plt.ylim(20, 2000)
    plt.tight_layout()
    plt.show()

def plot_notes(track_data):
    note_times = track_data["note_indices"][:, 0] * ANNOTATION_HOP
    note_freqs = FREQ_BINS_NOTES[track_data["note_indices"][:, 1]]

    plt.figure(figsize=(15, 3))
    plt.title("Adnotacje nut (notes_all)")
    plt.scatter(note_times, note_freqs, color="cyan", s=5)
    plt.ylabel("Częstotliwość [Hz]")
    plt.xlabel("Czas [s]")
    plt.ylim(20, 2000)
    plt.tight_layout()
    plt.show()

def plot_onsets(track_data):
    onset_times = track_data["onset_indices"][:, 0] * ANNOTATION_HOP
    onset_freqs = FREQ_BINS_NOTES[track_data["onset_indices"][:, 1]]

    plt.figure(figsize=(15, 3))
    plt.title("Onsety")
    plt.scatter(onset_times, onset_freqs, color="lime", marker="x", s=10)
    plt.ylabel("Częstotliwość [Hz]")
    plt.xlabel("Czas [s]")
    plt.ylim(20, 2000)
    plt.tight_layout()
    plt.show()

def plot_contours(track_data):
    contour_times = track_data["contour_indices"][:, 0] * ANNOTATION_HOP
    contour_freqs = FREQ_BINS_CONTOURS[track_data["contour_indices"][:, 1]]

    plt.figure(figsize=(15, 3))
    plt.title("Kontury częstotliwości (multif0)")
    plt.scatter(contour_times, contour_freqs, color="orange", s=1, alpha=0.5)
    plt.ylabel("Częstotliwość [Hz]")
    plt.xlabel("Czas [s]")
    plt.ylim(20, 2000)
    plt.tight_layout()
    plt.show()

plot_cqt(track_data["audio"])
plot_notes(track_data)
plot_onsets(track_data)
plot_contours(track_data)

### 2.4. Przygotowanie danych treningowych

In [ ]:
from typing import Tuple, Dict, List

def sparse_to_dense(indices: np.array, values: np.array, shape: Tuple[int, int]) -> np.array:
    """Konwertuj rzadką reprezentację adnotacji na gęstą macierz."""
    dense = np.zeros(shape, dtype=np.float32)
    for i, (t, f) in enumerate(indices):
        if t < shape[0] and f < shape[1]:
            dense[t, f] = values[i]
    return dense

def trim_time(data: np.ndarray, start: float, duration: float, sr: int) -> np.ndarray:
    """
    Wytnij fragment danych audio lub adnotacji.
    
    Args:
        data: Dane do przycięcia (audio lub adnotacje)
        start: Czas początkowy w sekundach
        duration: Długość fragmentu w sekundach
        sr: Częstotliwość próbkowania (dla audio) lub FPS (dla adnotacji)
    """
    n_start = int(np.round(start * sr))
    n_duration = int(np.ceil(duration * sr))
    end = min(n_start + n_duration, data.shape[0])
    return data[n_start:end]

def prepare_training_example(track_data: Dict, start_time: float) -> Dict:
    """
    Przygotuj pojedynczy przykład treningowy z określonego momentu w utworze.
    
    Args:
        track_data: Dane utworu z load_guitarset_track()
        start_time: Czas początkowy przykładu w sekundach
    """
    # Wytnij fragment audio
    audio_window = trim_time(
        track_data['audio'],
        start_time,
        AUDIO_WINDOW_LENGTH,
        AUDIO_SAMPLE_RATE
    )
    
    # Upewnij się, że okno ma odpowiednią długość
    if len(audio_window) < AUDIO_N_SAMPLES:
        padding = AUDIO_N_SAMPLES - len(audio_window)
        audio_window = np.pad(audio_window, (0, padding), mode='constant')
    
    # Konwertuj adnotacje na format gęsty
    dense_notes = sparse_to_dense(
        track_data['note_indices'],
        track_data['note_values'],
        (track_data['n_time_frames'], N_FREQ_BINS_NOTES)
    )
    dense_onsets = sparse_to_dense(
        track_data['onset_indices'],
        track_data['onset_values'],
        (track_data['n_time_frames'], N_FREQ_BINS_NOTES)
    )
    dense_contours = sparse_to_dense(
        track_data['contour_indices'],
        track_data['contour_values'],
        (track_data['n_time_frames'], N_FREQ_BINS_CONTOURS)
    )
    
    # Oblicz odpowiedni zakres ramek adnotacji
    frame_start = int(start_time * ANNOTATIONS_FPS)
    frame_end = frame_start + ANNOT_N_FRAMES
    
    # Wytnij fragmenty adnotacji
    note_window = dense_notes[frame_start:frame_end]
    onset_window = dense_onsets[frame_start:frame_end]
    contour_window = dense_contours[frame_start:frame_end]
    
    # Upewnij się, że okna mają odpowiednią długość
    if note_window.shape[0] < ANNOT_N_FRAMES:
        padding = ANNOT_N_FRAMES - note_window.shape[0]
        note_window = np.pad(note_window, ((0, padding), (0, 0)), mode='constant')
        onset_window = np.pad(onset_window, ((0, padding), (0, 0)), mode='constant')
        contour_window = np.pad(contour_window, ((0, padding), (0, 0)), mode='constant')
    
    return {
        'audio': audio_window,
        'notes': note_window,
        'onsets': onset_window,
        'contours': contour_window
    }

def generate_training_examples(track_data: Dict, examples_per_track: int = 20) -> List[Dict]:
    """
    Generuj przykłady treningowe z pojedynczego utworu.
    
    Args:
        track_data: Dane utworu z load_guitarset_track()
        examples_per_track: Liczba przykładów do wygenerowania z każdego utworu
    """
    examples = []
    duration = len(track_data['audio']) / AUDIO_SAMPLE_RATE
    max_start = duration - AUDIO_WINDOW_LENGTH
    
    if max_start <= 0:
        return examples  # Pomijamy zbyt krótkie utwory
    
    # Generuj losowe pozycje startowe
    start_times = np.random.uniform(0, max_start, size=examples_per_track)
    
    for start_time in start_times:
        example = prepare_training_example(track_data, start_time)
        examples.append(example)
    
    return examples

def create_dataset(guitarset_tracks: List, split: Split, examples_per_track: int = 20) -> List[Dict]:
    """
    Twórz zestaw danych dla określonego podziału (train/val/test).
    
    Args:
        guitarset_tracks: Lista obiektów track z GuitarSet
        split: Typ podziału (train/val/test)
        examples_per_track: Liczba przykładów na utwór
    """
    dataset = []
    
    for track in tqdm(guitarset_tracks, desc=f"Przetwarzanie {split.value}"):
        track_data = load_guitarset_track(track)
        examples = generate_training_examples(track_data, examples_per_track)
        dataset.extend(examples)
    
    return dataset

# Podział danych na zestawy treningowy, walidacyjny i testowy
track_ids = np.array(guitarset.track_ids)
np.random.shuffle(track_ids)

split_idx = {
    'train': int(0.7 * len(track_ids)),
    'val': int(0.85 * len(track_ids))
}

train_tracks = [guitarset.track(tid) for tid in track_ids[:split_idx['train']]]
val_tracks = [guitarset.track(tid) for tid in track_ids[split_idx['train']:split_idx['val']]]
test_tracks = [guitarset.track(tid) for tid in track_ids[split_idx['val']:]]

# Generowanie datasetów
train_dataset = create_dataset(train_tracks, Split.train)
val_dataset = create_dataset(val_tracks, Split.validation)
test_dataset = create_dataset(test_tracks, Split.test)

print(f"Liczba przykładów treningowych: {len(train_dataset)}")
print(f"Liczba przykładów walidacyjnych: {len(val_dataset)}")
print(f"Liczba przykładów testowych: {len(test_dataset)}")

## Funkcje potrzebne modelowi:

In [ ]:
class NormalizedLog(nn.Module):
    """
    Przekształca tensor wejściowy (batch, x, y, z) lub (batch, y, z)
    do skali dB i normalizuje do zakresu 0-1.
    Zakłada, że x=1, jeśli obecny.
    Dodaje 1e-10 do wartości wejściowych, aby uniknąć NaN.
    """

    def __init__(self):
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Obsługa kształtów (batch, 1, y, z) albo (batch, y, z)
        if x.ndim == 4:
            assert x.shape[1] == 1, f"Oczekiwano x=1 na drugiej osi, otrzymano {x.shape[1]}"
            x = x.squeeze(1)
        elif x.ndim != 3:
            raise ValueError(f"Obsługiwane są tylko tensory o rank 3 lub 4. Otrzymano: {x.shape}")

        # Oblicz moc (amplituda^2)
        power = x ** 2
        log_power = 10 * torch.log10(power + 1e-10)

        # Min/max normalizacja log-mocy do zakresu 0-1
        log_min = log_power.amin(dim=(1, 2), keepdim=True)
        log_offset = log_power - log_min
        log_max = log_offset.amax(dim=(1, 2), keepdim=True)

        log_norm = log_offset / (log_max + 1e-10)

        return log_norm

class HarmonicStacking(nn.Module):
    """
    Warstwa do harmonicznego stackowania.

    Wejście: (batch, time, freq, 1)
    Wyjście: (batch, time, n_output_freqs, len(harmonics))

    Argumenty:
        bins_per_semitone: liczba binów na półton (CQT resolution)
        harmonics: lista mnożników harmonicznych (np. [1.0, 2.0, 3.0])
        n_output_freqs: liczba wyjściowych częstotliwości (obcinamy na końcu)
    """

    def __init__(self, bins_per_semitone: int, harmonics: list[float], n_output_freqs: int):
        super().__init__()
        self.bins_per_semitone = bins_per_semitone
        self.harmonics = harmonics
        self.n_output_freqs = n_output_freqs

        # Preliczamy przesunięcia (w binach) dla każdej harmonicznej
        self.shifts = [
            int(round(12.0 * bins_per_semitone * math.log2(h))) for h in harmonics
        ]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: Tensor o kształcie (batch, time, freq, 1)
        """
        assert x.ndim == 4 and x.shape[-1] == 1, f"Oczekiwany kształt (B, T, F, 1), otrzymano {x.shape}"

        stacked = []

        for shift in self.shifts:
            if shift == 0:
                shifted = x
            elif shift > 0:
                shifted = F.pad(x[:, :, shift:, :], (0, 0, 0, shift))  # pad na końcu
            else:
                shifted = F.pad(x[:, :, :shift, :], (0, 0, -shift, 0))  # pad na początku

            stacked.append(shifted)

        # Połączamy w kanałach (na końcu ostatniego wymiaru)
        x_harmonics = torch.cat(stacked, dim=-1)  # (B, T, F, n_harmonics)

        # Obcinamy do n_output_freqs (wymiar częstotliwości)
        x_harmonics = x_harmonics[:, :, :self.n_output_freqs, :]

        return x_harmonics


class FlattenFreqCh(nn.Module):
    """
    Spłaszcza oś częstotliwości i kanałów do jednej osi.

    Wejście:  (batch, time, freq, channels)
    Wyjście: (batch, time, freq * channels)
    """

    def __init__(self):
        super().__init__()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert x.ndim == 4, f"Oczekiwany tensor 4D, otrzymano {x.ndim}D o kształcie {x.shape}"
        batch, time, freq, ch = x.shape
        return x.view(batch, time, freq * ch)

from nnAudio.Spectrogram import CQT1992v2

class CQT2010v2(nn.Module):
    def __init__(
        self,
        sr=22050,
        hop_length=512,
        fmin=32.70,
        fmax=None,
        n_bins=84,
        bins_per_octave=12,
        filter_scale=1,
        norm=1,
        window='hann',
        pad_mode='reflect',
        output_format='Magnitude',
        trainable=False,
        device=None,
    ):
        super().__init__()
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.output_format = output_format.lower()
        
        self.cqt_layer = CQT1992v2(
            sr=sr,
            hop_length=hop_length,
            fmin=fmin,
            fmax=fmax,
            n_bins=n_bins,
            bins_per_octave=bins_per_octave,
            filter_scale=filter_scale,
            norm=norm,
            window=window,
            pad_mode=pad_mode,
            output_format=self.output_format,
            trainable=trainable,
        ).to(self.device)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = x.to(self.device)
        output = self.cqt_layer(x)
        if output is None:
            print("CQT1992v2 zwrócił None!")
        return output


In [ ]:
MAX_N_SEMITONES = int(np.floor(12.0 * np.log2(0.5 * AUDIO_SAMPLE_RATE / ANNOTATIONS_BASE_FREQUENCY)))


def _initializer(tensor):
    """Equivalent of tf.keras.initializers.VarianceScaling(scale=2.0, mode='fan_avg')"""
    if tensor is not None:
        nn.init.kaiming_uniform_(tensor, a=np.sqrt(5))


class BasicPitchModel(nn.Module):
    def __init__(self, n_harmonics=8, n_filters_contour=32, n_filters_onsets=32, n_filters_notes=32, no_contours=False):
        super().__init__()
        self.no_contours = no_contours

        # CQT params
        n_semitones = min(
            int(np.ceil(12.0 * np.log2(n_harmonics)) + ANNOTATIONS_N_SEMITONES),
            MAX_N_SEMITONES
        )
        self.cqt = CQT2010v2(
            sr=AUDIO_SAMPLE_RATE,
            hop_length=FFT_HOP,
            fmin=ANNOTATIONS_BASE_FREQUENCY,
            n_bins=n_semitones * CONTOURS_BINS_PER_SEMITONE,
            bins_per_octave=12 * CONTOURS_BINS_PER_SEMITONE
        )
        self.norm_log = NormalizedLog()

        # Harmonic stacking
        self.harmonic_stack = HarmonicStacking(
            CONTOURS_BINS_PER_SEMITONE,
            [0.5] + list(range(1, n_harmonics)) if n_harmonics > 1 else [1],
            N_FREQ_BINS_CONTOURS
        )

        # Contour path
        self.contour_conv1 = nn.Conv2d(1, n_filters_contour, kernel_size=5, padding=2)
        self.contour_bn1 = nn.BatchNorm2d(n_filters_contour)

        self.contour_conv2 = nn.Conv2d(n_filters_contour, 8, kernel_size=(3, 39), padding=(1, 19))
        self.contour_bn2 = nn.BatchNorm2d(8)

        if not self.no_contours:
            self.contour_out = nn.Conv2d(8, 1, kernel_size=5, padding=2)

        # Notes path
        contour_in_ch = 1 if not no_contours else 8
        self.contour_to_notes = nn.Conv2d(contour_in_ch, n_filters_notes, kernel_size=7, stride=(1, 3), padding=3)
        self.note_out = nn.Conv2d(n_filters_notes, 1, kernel_size=(7, 3), padding=(3, 1))

        # Onsets path
        self.onset_conv = nn.Conv2d(1, n_filters_onsets, kernel_size=5, stride=(1, 3), padding=2)
        self.onset_bn = nn.BatchNorm2d(n_filters_onsets)
        self.onset_merge = nn.Conv2d(n_filters_onsets + 1, 1, kernel_size=3, padding=1)

        # Flatten
        self.flatten = FlattenFreqCh()

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d):
            _initializer(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # x shape: (B, T)
        x = x.unsqueeze(1)  # (B, 1, T)
        x = self.cqt(x)     # (B, F, T)
        x = self.norm_log(x)
        x = x.unsqueeze(1)  # (B, 1, F, T)

        x_harmonic = self.harmonic_stack(x)  # (B, H, F, T)

        # --- Contours ---
        x_cont = F.relu(self.contour_bn1(self.contour_conv1(x_harmonic)))
        x_cont = F.relu(self.contour_bn2(self.contour_conv2(x_cont)))

        if not self.no_contours:
            x_cont_out = torch.sigmoid(self.contour_out(x_cont))
            x_cont_reduced = x_cont_out
        else:
            x_cont_out = None
            x_cont_reduced = x_cont

        # --- Notes ---
        x_notes = F.relu(self.contour_to_notes(x_cont_reduced))
        x_notes_out = torch.sigmoid(self.note_out(x_notes))

        # --- Onsets ---
        x_onset = F.relu(self.onset_bn(self.onset_conv(x_harmonic)))
        x_onset = torch.cat([x_notes_out, x_onset], dim=1)
        x_onset = torch.sigmoid(self.onset_merge(x_onset))

        return {
            "contour": self.flatten(x_cont_out) if x_cont_out is not None else None,
            "note": self.flatten(x_notes_out),
            "onset": self.flatten(x_onset),
        }

## Pętla treningowa

In [ ]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import time

# 1. Klasa Dataset dla przygotowanych danych
class GuitarSetDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sample = self.data[idx]
        # Konwersja numpy arrays do torch tensors
        audio = torch.from_numpy(sample['audio']).float()
        notes = torch.from_numpy(sample['notes']).float()
        onsets = torch.from_numpy(sample['onsets']).float()
        contours = torch.from_numpy(sample['contours']).float()
        return audio, notes, onsets, contours

# 2. Inicjalizacja DataLoaderów
batch_size = 16
train_ds = GuitarSetDataset(train_dataset)
val_ds = GuitarSetDataset(val_dataset)
test_ds = GuitarSetDataset(test_dataset)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=4)

# 3. Inicjalizacja modelu i optymalizatora
model = BasicPitchModel(no_contours=False).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Funkcje straty (bez zmian)
def weighted_bce_loss(y_pred, y_true, positive_weight=0.5):
    """Binary cross-entropy z wagą dla klasy pozytywnej"""
    loss = F.binary_cross_entropy(y_pred, y_true, reduction='none')
    weight = torch.ones_like(y_true) * (1 - positive_weight)
    weight[y_true > 0.5] = positive_weight
    return (loss * weight).mean()

def compute_loss(outputs, targets, weighted_onset_loss=True, positive_onset_weight=0.5):
    """Oblicza całkowitą stratę dla wszystkich wyjść modelu"""
    losses = {}
    
    # Strata dla nut
    losses['note'] = F.binary_cross_entropy(outputs['note'], targets['note'])
    
    # Strata dla onsetów
    if weighted_onset_loss:
        losses['onset'] = weighted_bce_loss(outputs['onset'], targets['onset'], positive_onset_weight)
    else:
        losses['onset'] = F.binary_cross_entropy(outputs['onset'], targets['onset'])
    
    # Strata dla konturów (jeśli są używane)
    if outputs['contour'] is not None:
        losses['contour'] = F.binary_cross_entropy(outputs['contour'], targets['contour'])
    else:
        losses['contour'] = torch.tensor(0.0, device=device)
    
    # Suma strat z równymi wagami
    total_loss = sum(losses.values())
    
    return total_loss, losses

# 5. Zmodyfikowana pętla treningowa
def train_model(model, train_loader, val_loader, epochs=50, patience=5):
    best_val_loss = float('inf')
    patience_counter = 0
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'note_loss': [],
        'onset_loss': [],
        'contour_loss': []
    }
    
    for epoch in range(epochs):
        start_time = time.time()
        
        # Tryb treningowy
        model.train()
        train_loss = 0.0
        
        for audio, notes, onsets, contours in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
            audio = audio.to(device)
            targets = {
                'note': notes.to(device),
                'onset': onsets.to(device),
                'contour': contours.to(device)
            }
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(audio)
            
            # Oblicz stratę
            loss, loss_components = compute_loss(
                outputs, 
                targets,
                weighted_onset_loss=True,
                positive_onset_weight=0.5
            )
            
            # Backward pass i optymalizacja
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Średnia strata treningowa
        train_loss /= len(train_loader)
        history['train_loss'].append(train_loss)
        
        # Walidacja
        model.eval()
        val_loss = 0.0
        note_loss = 0.0
        onset_loss = 0.0
        contour_loss = 0.0
        
        with torch.no_grad():
            for audio, notes, onsets, contours in val_loader:
                audio = audio.to(device)
                targets = {
                    'note': notes.to(device),
                    'onset': onsets.to(device),
                    'contour': contours.to(device)
                }
                
                outputs = model(audio)
                loss, loss_components = compute_loss(
                    outputs, 
                    targets,
                    weighted_onset_loss=True,
                    positive_onset_weight=0.5
                )
                
                val_loss += loss.item()
                note_loss += loss_components['note'].item()
                onset_loss += loss_components['onset'].item()
                contour_loss += loss_components['contour'].item()
        
        # Średnie straty walidacyjne
        val_loss /= len(val_loader)
        note_loss /= len(val_loader)
        onset_loss /= len(val_loader)
        contour_loss /= len(val_loader)
        
        history['val_loss'].append(val_loss)
        history['note_loss'].append(note_loss)
        history['onset_loss'].append(onset_loss)
        history['contour_loss'].append(contour_loss)
        
        # Wydrukuj statystyki
        epoch_time = time.time() - start_time
        print(f'\nEpoch {epoch+1}/{epochs} - {epoch_time:.1f}s')
        print(f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')
        print(f'Note Loss: {note_loss:.4f} | Onset Loss: {onset_loss:.4f} | Contour Loss: {contour_loss:.4f}')
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'\nEarly stopping after {epoch+1} epochs')
                break
    
    return history

# 6. Uruchomienie treningu
history = train_model(model, train_loader, val_loader, epochs=100)

# 7. Funkcja do wizualizacji wyników (bez zmian)
def plot_training_history(history):
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history['note_loss'], label='Note Loss')
    plt.plot(history['onset_loss'], label='Onset Loss')
    plt.plot(history['contour_loss'], label='Contour Loss')
    plt.title('Component Validation Losses')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

plot_training_history(history)